# SQL Database Creation & Analysis


### Workflow

Cleaned CSVs
→ PostgreSQL Database
→ Tables & Constraints
→ Data Loading
→ Database Validation
→ SQL Analysis
→ Business Findings

##  Cart2Insights Key Structure

| Table | Primary Key | Foreign Key(s) | References |
|---|---|---|---|
| customers | `customer_id` | — | — |
| orders | `order_id` | `customer_id` | `customers.customer_id` |
| products | `product_id` | `product_category_name` | `product_category_name_translation.product_category_name` |
| sellers | `seller_id` | — | — |
| order_items | `(order_id, order_item_id)` | `order_id`, `product_id`, `seller_id` | `orders`, `products`, `sellers` |
| order_payments | `(order_id, payment_sequential)` | `order_id` | `orders.order_id` |
| order_reviews | `(review_id, order_id)` | `order_id` | `orders.order_id` |
| geolocation | No source-aligned primary key | — | — |
| product_category_name_translation | `product_category_name` | — | — |

##  Important Data-Driven Exceptions

### `order_reviews.review_id`

The ER diagram represents `review_id` as a primary key.

During data-quality analysis, we found that `review_id` is not unique
in the supplied dataset.

However, `(review_id, order_id)` is unique.

Therefore, the physical SQL schema uses:

`PRIMARY KEY (review_id, order_id)`

rather than forcing `review_id` to be the sole primary key.

### `geolocation_zip_code_prefix`

The ER diagram represents `geolocation_zip_code_prefix` as a key.

However, our data-quality analysis showed that multiple geolocation
observations can have the same ZIP prefix.

Therefore, `geolocation_zip_code_prefix` cannot safely be used as a
primary key in the physical database.

We retain the geolocation table at its actual observation grain and
use the ZIP column for indexing/filtering rather than imposing an
invalid uniqueness constraint.

## Foreign Keys from the ER Diagram — What We Enforce and Why

The ER diagram represents the intended relationships between the
tables. Before implementing these relationships as SQL foreign-key
constraints, we validated the actual supplied data during the data
quality and cleaning phases.

A foreign key can safely be enforced when the referenced column in the
parent table uniquely identifies a row and the child values are valid
references to that parent.

Therefore, our physical SQL schema follows:

ER Diagram
→ Actual Data Validation
→ Cleaning Decisions
→ SQL Constraints

### Foreign Keys We Enforce

| Child Table | Foreign Key | Parent Table | Parent Key | Reason |
|---|---|---|---|---|
| `orders` | `customer_id` | `customers` | `customer_id` | Parent key is unique and 100% of child keys matched. |
| `products` | `product_category_name` | `product_category_name_translation` | `product_category_name` | Translation category is unique and product categories can be mapped to it. |
| `order_items` | `order_id` | `orders` | `order_id` | `order_id` is unique in `orders` and all order-item references are valid. |
| `order_items` | `product_id` | `products` | `product_id` | `product_id` is unique in `products` and all references are valid. |
| `order_items` | `seller_id` | `sellers` | `seller_id` | `seller_id` is unique in `sellers` and all references are valid. |
| `order_payments` | `order_id` | `orders` | `order_id` | `order_id` is unique in `orders` and payment references are valid. |
| `order_reviews` | `order_id` | `orders` | `order_id` | `order_id` is unique in `orders` and review references are valid. |

These relationships were supported by the foreign-key validation
performed during data-quality analysis.

---

### ER Relationships We Do NOT Enforce as SQL Foreign Keys

#### Customer ZIP → Geolocation ZIP

The ER diagram represents:

`customers.customer_zip_code_prefix`
→ `geolocation.geolocation_zip_code_prefix`

However, the actual `geolocation` table does not contain one unique row
per ZIP prefix.

During data-quality analysis, we found:

- approximately 1 million geolocation rows
- repeated `geolocation_zip_code_prefix` values
- multiple geolocation observations for the same ZIP prefix

Therefore, `geolocation_zip_code_prefix` cannot safely be used as the
primary key referenced by a normal SQL foreign key.

We therefore do NOT create:

`customers.customer_zip_code_prefix`
→ `geolocation.geolocation_zip_code_prefix`

as an enforced foreign-key constraint.

The geographic relationship is still useful for analysis, but any join
using ZIP must account for the multiple geolocation observations per ZIP.

---

#### Seller ZIP → Geolocation ZIP

The ER diagram also represents:

`sellers.seller_zip_code_prefix`
→ `geolocation.geolocation_zip_code_prefix`

The same issue applies here.

Because `geolocation_zip_code_prefix` is not unique in the actual
dataset, it cannot serve as the referenced candidate key for a standard
foreign-key constraint.

Therefore, this relationship is also not enforced as a SQL FK.

The ZIP relationship can still be used analytically, but it must be
handled as a one-to-many relationship rather than assuming one
geolocation row per ZIP.

---

### Why We Do Not Blindly Copy the ER Diagram

The ER diagram is the conceptual model supplied for the project.

The SQL database is the physical implementation.

In this project, some assumptions in the conceptual model do not match
the actual supplied data.

For example:

ER Diagram:
`geolocation_zip_code_prefix` is shown as a key.

Actual data:
`geolocation_zip_code_prefix` is repeated across many rows.

Therefore, we do not create a primary key or foreign-key constraint
that the actual data cannot satisfy.

This is an intentional, data-driven schema decision rather than a
missing relationship.

---

.

## Product Category Translation — Data Validation

The ER diagram represents `products.product_category_name` as a foreign
key to `product_category_name_translation.product_category_name`.

The translation table contains 71 unique categories. However, the
products table contains 73 distinct non-null product categories.

Two categories are present in products but absent from the translation
table:

- `pc_gamer`
- `portateis_cozinha_e_preparadores_de_alimentos`

Therefore, the relationship is retained for analytical joins but is not
enforced as a physical foreign-key constraint. We do not delete these
products or invent translations.

This is a data-driven deviation from the conceptual ER diagram.